# ReAct Agents with `create_react_agent`
## Reason + Act — The Loop That Powers Modern Agents
⏱ ~10 min

**ReAct** (Reason + Act) is the foundational loop behind every tool-calling agent. The model reasons about what to do, selects a tool, observes the result, and repeats until it has a final answer. LangGraph's `create_react_agent` assembles this entire cycle — LLM, tools, memory, and state graph — in a single function call.

By the end of this workshop you will understand *why* the ReAct loop works, *how* `create_react_agent` wires the pieces together internally, and *how* to build agents that handle math, product Q&A, multi-turn memory, and streaming output.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — ReAct loop, how tool selection works, key vocabulary |
| 2 | **Setup** — install, API key, imports |
| 3 | **First Agent** — `@tool`, docstrings, `create_react_agent` |
| 4 | **Streaming** — `stream_mode="values"`, reading message traces |
| 5 | **Multi-step chaining** — agent calls multiple tools in sequence |
| 6 | **Memory** — `MemorySaver` + `thread_id` for multi-turn conversations |
| 7 | **Real-world agent** — PDF retrieval + CSV pricing + chat persona |
| 8 | **Debugging** — printing state, tracing tool calls, common failures |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with a `.venv` containing `requirements.txt` (local), **or** Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- For the real-world agent (Part 7): `data/product/Laptop pricing.csv` and `data/product/Laptop product descriptions.pdf`

### Key References
> Yao, S., Zhao, J., Yu, D., et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629
>
> Wei, J., Wang, X., Schuurmans, D., et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models.* NeurIPS 2022. https://arxiv.org/abs/2201.11903
>
> LangGraph `create_react_agent` reference: https://langchain-ai.github.io/langgraph/reference/prebuilt/#langgraph.prebuilt.chat_agent_executor.create_react_agent
>
> LangChain `@tool` decorator: https://python.langchain.com/docs/concepts/tools/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "langchain-community",
            "langchain-text-splitters",
            "langgraph",
            "chromadb",
            "python-dotenv",
            "pypdf",
            "pandas",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY not found or invalid. "
    "Local: add it to .env. Colab: add it in the Secrets panel."
)
print(f"OPENAI_API_KEY present: True  ({key[:8]}...)")

---

## Part 1 — Concepts: The ReAct Loop

---

### The Problem ReAct Solves

A plain LLM is a *read-only* oracle: it can reason about information in its weights and context window, but it cannot **take actions** — look up a live price, run a calculation, query a database, or retrieve a document.

**Tool calling** solves this by letting the LLM output a structured "call this function with these arguments" response instead of plain text. The application runs the function, adds the result to the conversation, and lets the model continue.

**ReAct** formalizes this into a loop: **Reason** about what to do → **Act** (call a tool) → **Observe** the result → repeat until done.

---

### ReAct Architecture Diagram

```
User: "What is 3 × 2, then add 5?"
  │
  ▼
┌─────────────────────────────────────────────────────────────┐
│                    LangGraph State Graph                    │
│                                                             │
│  ┌──────────┐    tool_calls?     ┌──────────────────────┐  │
│  │  agent   │ ─────────────────► │   tools (executor)   │  │
│  │  (LLM)  │                    │  find_product(3, 2)   │  │
│  └──────────┘ ◄───────────────── │  → 6                 │  │
│       │        ToolMessage(6)    └──────────────────────┘  │
│       │                                                     │
│       │  [Thought: I have 6, now add 5]                     │
│       │                                                     │
│  ┌──────────┐    tool_calls?     ┌──────────────────────┐  │
│  │  agent   │ ─────────────────► │   tools (executor)   │  │
│  │  (LLM)  │                    │  find_sum(6, 5)       │  │
│  └──────────┘ ◄───────────────── │  → 11                │  │
│       │        ToolMessage(11)   └──────────────────────┘  │
│       │                                                     │
│       │  [No more tool_calls → final answer]                │
│       ▼                                                     │
│  AIMessage("The answer is 11.")                             │
└─────────────────────────────────────────────────────────────┘
                        │
                        ▼
               User receives answer
```

The loop runs inside the graph until the LLM produces a message with **no tool calls** — that is the termination condition.

---

### ReAct vs Chain-of-Thought vs Plan-Execute

| Approach | How it works | Tool use | Memory | Best for |
|----------|-------------|----------|--------|----------|
| **Chain-of-Thought (CoT)** | LLM reasons step-by-step in text only | None | None | Pure reasoning tasks |
| **ReAct** | Interleaves reasoning with real tool calls | Yes (dynamic) | Optional | Most agent tasks |
| **Plan-Execute** | Plan all steps up front, then execute in batch | Yes (planned) | Optional | Long, predictable workflows |
| **ReAct + Memory** | ReAct loop + conversation history | Yes | Yes | Multi-turn assistants |

ReAct is the default choice because it is flexible enough for most tasks without requiring an explicit planning stage.

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **`@tool`** | LangChain decorator that converts a Python function into a tool the LLM can call |
| **Docstring** | The natural-language description the LLM reads to decide *when* to call a tool |
| **Tool schema** | The JSON schema derived from the function signature that tells the LLM *how* to call the tool |
| **`AIMessage`** | A message from the LLM — may contain `tool_calls` or plain `content` |
| **`ToolMessage`** | The result returned after a tool executes, keyed to the tool call ID |
| **`MemorySaver`** | LangGraph checkpoint backend that persists state between invocations |
| **`thread_id`** | Unique identifier that scopes a `MemorySaver` checkpoint to one conversation |
| **`stream_mode="values"`** | Streaming mode that yields the full agent state after each graph step |

---

## Part 2 — Setup: Imports

---

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import uuid

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

# ── Default LLM ───────────────────────────────────────────────────────────────
# gpt-4o-mini is fast and cheap — good for learning and experimentation.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("Imports OK.")
print(f"Model: {llm.model_name}")

---

## Part 3 — First Agent: `@tool` and `create_react_agent`

---

### How Tool Selection Works

When the LLM receives a query it looks at the list of available tools and decides which one (if any) to call. The decision is based entirely on:

1. **The tool's docstring** — the natural-language description of what the tool does and when to use it
2. **The tool's parameter schema** — derived from the Python function signature (name, type, default values)

This means **docstring quality directly determines agent quality**. A vague or misleading docstring causes the LLM to call the wrong tool or skip it entirely.

```python
# Bad docstring — LLM may ignore this tool or misuse it
@tool
def calculate(x: int, y: int) -> int:
    """Do math."""
    return x + y

# Good docstring — LLM knows exactly when and how to use this
@tool
def find_sum(x: int, y: int) -> int:
    """Add two integers and return their sum. Use this tool whenever
    the user needs to compute the sum or addition of two numbers."""
    return x + y
```

In [ ]:
# ─── 3-A: Define tools with @tool ────────────────────────────────────────────
# Each tool is a plain Python function.
# The @tool decorator wraps it in a LangChain Tool object.
# The docstring is sent to the LLM as the tool description.
# The type annotations define the JSON schema for the function arguments.

@tool
def find_sum(x: int, y: int) -> int:
    """
    Add two numbers and return their sum.
    Use this tool whenever the user needs to compute the sum or
    addition of two integer numbers.
    """
    return x + y


@tool
def find_product(x: int, y: int) -> int:
    """
    Multiply two numbers and return their product.
    Use this tool whenever the user needs to compute the product
    or multiplication of two integer numbers.
    """
    return x * y


# Inspect what the LLM sees for each tool
print("Tool name:", find_sum.name)
print("Tool description:", find_sum.description)
print("Tool args schema:", find_sum.args)
print()
print("Tool name:", find_product.name)
print("Tool description:", find_product.description)
print("Tool args schema:", find_product.args)

In [ ]:
# ─── 3-B: Build the agent with create_react_agent ────────────────────────────
# create_react_agent wires together:
#   - model: the LLM that reasons and calls tools
#   - tools: the list of available tools
#   - prompt: a SystemMessage that sets the agent's persona
#   - checkpointer: optional MemorySaver for multi-turn memory
#
# It returns a CompiledGraph — a runnable LangGraph state machine.

system_prompt = SystemMessage(
    "You are a Math assistant. Solve problems using ONLY the available tools. "
    "Do NOT compute answers yourself — always use a tool."
)

math_agent = create_react_agent(
    model=llm,
    tools=[find_sum, find_product],
    prompt=system_prompt,
    checkpointer=MemorySaver(),
)

# The agent is a compiled LangGraph graph. Inspect its nodes.
print("Agent type:", type(math_agent).__name__)
print("Graph nodes:", list(math_agent.get_graph().nodes.keys()))

In [ ]:
# ─── 3-C: Invoke the agent — simplest usage ───────────────────────────────────
# .invoke() blocks until the agent reaches a final answer.
# Returns a dict with key 'messages' containing the full message history.

config = {"configurable": {"thread_id": "demo-1"}}

result = math_agent.invoke(
    {"messages": [HumanMessage("What is the sum of 2 and 3?")]},
    config,
)

print("Full message history:")
for msg in result["messages"]:
    role = type(msg).__name__
    content = str(msg.content)[:120] if msg.content else ""
    # AIMessage may also have tool_calls — show them if present
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"  [{role}] tool_calls={msg.tool_calls}")
    else:
        print(f"  [{role}] {content}")

print()
print("Final answer:", result["messages"][-1].content)

---

## Part 4 — Streaming: Watching the ReAct Loop Step by Step

---

`.invoke()` blocks until completion — you see nothing until the agent finishes. For longer tasks or UX reasons, use `.stream()` instead. It yields the agent's state after every graph step.

### `stream_mode` options

| Mode | Yields | Best for |
|------|--------|---------|
| `"values"` | Full state dict after each step | Seeing the latest message at each step |
| `"updates"` | Only the delta (what changed) per step | Efficient processing of large state |
| `"messages"` | Token-by-token message stream | Real-time streaming UX |

Use `stream_mode="values"` for debugging — it gives you the complete picture at every step.

In [ ]:
# ─── 4-A: Stream a single tool call ──────────────────────────────────────────
# Each iteration of the loop yields the agent state after one graph step.
# Printing the last message each iteration reveals the Thought→Action→Observation cycle.

config_s1 = {"configurable": {"thread_id": "stream-demo-1"}}

print("Q: What is the sum of 7 and 9?\n")
print("--- Step-by-step trace ---")

for step_num, state in enumerate(math_agent.stream(
    {"messages": [HumanMessage("What is the sum of 7 and 9?")]},
    config_s1,
    stream_mode="values",
)):
    last_msg = state["messages"][-1]
    role = type(last_msg).__name__
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        tc = last_msg.tool_calls[0]
        print(f"  Step {step_num} [{role}]  → calling {tc['name']}({tc['args']})")
    elif isinstance(last_msg, ToolMessage):
        print(f"  Step {step_num} [{role}]  → tool result: {last_msg.content}")
    else:
        print(f"  Step {step_num} [{role}]  → {str(last_msg.content)[:100]}")

In [ ]:
# ─── 4-B: Pretty-print with .pretty_print() ──────────────────────────────────
# LangChain messages have a .pretty_print() method — useful for quick debugging.

config_s2 = {"configurable": {"thread_id": "stream-demo-2"}}

print("Q: What is 6 times 7?\n")
for state in math_agent.stream(
    {"messages": [HumanMessage("What is 6 times 7?")]},
    config_s2,
    stream_mode="values",
):
    state["messages"][-1].pretty_print()
    print("───")

---

## Part 5 — Multi-Step Tool Chaining

---

A single query can require the agent to call multiple tools in sequence, using the output of one as the input to the next. The agent decides the order based on its reasoning — you don't have to specify it explicitly.

This is the key power of the ReAct pattern: the model reasons about dependencies between tool calls and handles them automatically.

In [ ]:
# ─── 5-A: Two-tool chain — multiply then add ─────────────────────────────────
# Expected behavior:
#   Step 1: find_product(3, 2) → 6
#   Step 2: find_sum(6, 5) → 11
#   Final:  "The answer is 11."

config_c1 = {"configurable": {"thread_id": "chain-1"}}

print("Q: What is 3 multiplied by 2, then add 5?\n")
result = math_agent.invoke(
    {"messages": [HumanMessage("What is 3 multiplied by 2, then add 5?")]},
    config_c1,
)

print("Full trace:")
for msg in result["messages"]:
    role = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  [{role}]  TOOL CALL: {tc['name']}({tc['args']})")
    elif isinstance(msg, ToolMessage):
        print(f"  [{role}]  RESULT: {msg.content}")
    elif isinstance(msg, AIMessage) and msg.content:
        print(f"  [{role}]  {msg.content}")
    elif isinstance(msg, HumanMessage):
        print(f"  [{role}]  {msg.content}")

print()
print("Final answer:", result["messages"][-1].content)

In [ ]:
# ─── 5-B: Count tool calls in a response ─────────────────────────────────────
# Useful diagnostic: see how many times each tool was called.

from collections import Counter

config_c2 = {"configurable": {"thread_id": "chain-2"}}

result2 = math_agent.invoke(
    {"messages": [HumanMessage(
        "What is (4 plus 6) multiplied by (3 plus 7)?"
    )]},
    config_c2,
)

tool_call_counts = Counter()
for msg in result2["messages"]:
    if hasattr(msg, "tool_calls"):
        for tc in msg.tool_calls:
            tool_call_counts[tc["name"]] += 1

print("Tool calls made:", dict(tool_call_counts))
print("Total steps:", len(result2["messages"]))
print("Final answer:", result2["messages"][-1].content)

---

## Part 6 — Memory: `MemorySaver` and `thread_id`

---

### How Memory Works in LangGraph

Without memory, every `.invoke()` call starts with a blank state. The agent has no knowledge of previous turns.

**`MemorySaver`** is a LangGraph checkpoint backend that persists the full agent state (message history) to an in-process dictionary. Pass it as `checkpointer` to `create_react_agent`.

**`thread_id`** scopes the checkpoint to a specific conversation. Use a unique `thread_id` per user session.

```
                  MemorySaver (in-process dict)
                  ┌─────────────────────────────────┐
thread_id="user-A"│  [HumanMsg, AIMsg, ToolMsg, ...]│
thread_id="user-B"│  [HumanMsg, AIMsg, ToolMsg, ...]│
thread_id="user-C"│  [HumanMsg, AIMsg, ToolMsg, ...]│
                  └─────────────────────────────────┘
```

Each `invoke()` call with the same `thread_id` appends to the existing history — the agent can refer back to earlier messages.

In [ ]:
# ─── 6-A: Multi-turn memory with same thread_id ───────────────────────────────
# Turn 1: ask a question. Turn 2: refer back to the answer.
# The agent should remember the result from Turn 1 without being told.

config_mem = {"configurable": {"thread_id": "memory-demo"}}

r1 = math_agent.invoke(
    {"messages": [HumanMessage("What is 7 plus 8?")]},
    config_mem,
)
print("Turn 1:", r1["messages"][-1].content)

# Refer to the previous result — agent must use memory
r2 = math_agent.invoke(
    {"messages": [HumanMessage("Multiply that result by 3.")]},
    config_mem,
)
print("Turn 2:", r2["messages"][-1].content)

# Check total message count: should include messages from both turns
print(f"Total messages in thread: {len(r2['messages'])}")

In [ ]:
# ─── 6-B: Thread isolation — fresh thread has no memory ──────────────────────
# A new thread_id starts with an empty history.
# The agent should not know "that result" since it's a fresh conversation.

config_fresh = {"configurable": {"thread_id": "fresh-thread-" + str(uuid.uuid4())}}

r3 = math_agent.invoke(
    {"messages": [HumanMessage("Multiply that result by 3.")]},
    config_fresh,
)
print("Fresh thread (no prior context):", r3["messages"][-1].content)
print("Expected: agent asks for clarification or cannot compute")

In [ ]:
# ─── 6-C: Two users, two threads, isolated conversations ──────────────────────
# Simulates a real app where two users ask different questions.
# Their conversations never bleed into each other.

def chat(agent, config, message: str) -> str:
    """Send one message to the agent and return the final text response."""
    result = agent.invoke(
        {"messages": [HumanMessage(message)]},
        config,
    )
    return result["messages"][-1].content


config_user1 = {"configurable": {"thread_id": "user-1"}}
config_user2 = {"configurable": {"thread_id": "user-2"}}

print("USER 1 Turn 1:", chat(math_agent, config_user1, "What is 10 plus 5?"))
print("USER 2 Turn 1:", chat(math_agent, config_user2, "What is 4 times 6?"))

# Each user refers to "that result" — they should get different answers
print("USER 1 Turn 2:", chat(math_agent, config_user1, "Add 2 to that result."))
print("USER 2 Turn 2:", chat(math_agent, config_user2, "Add 2 to that result."))

---

## Part 7 — Real-World Agent: Product Q&A with PDF + CSV Tools

---

The math agent demonstrates the mechanics. This section builds a realistic **product Q&A assistant** that:
- Retrieves laptop features from a **PDF** (semantic search via ChromaDB)
- Looks up laptop prices from a **CSV** (exact match via pandas)
- Maintains a professional persona and full conversation memory

This is the same agent from `src/agent/product_qna.py` — explained step by step.

### Tool Architecture

```
User: "What are the features of SpectraBook and how much does it cost?"
  │
  ▼
┌─────────────────────────────────────────────────────┐
│                  product_qna agent                  │
│                                                     │
│  ┌──────────────────────┐  ┌─────────────────────┐ │
│  │  Get_Product_Features│  │  get_laptop_price   │ │
│  │  PDF → ChromaDB      │  │  CSV → pandas match │ │
│  │  semantic k=1        │  │  prefix match       │ │
│  └──────────────────────┘  └─────────────────────┘ │
└─────────────────────────────────────────────────────┘
  │
  ▼
"SpectraBook has a 12-core CPU, 32GB RAM... It costs $1,499."
```

In [ ]:
# ─── 7-A: CSV pricing tool ────────────────────────────────────────────────────
# Reads a CSV with columns 'Name' and 'Price'.
# Performs a case-insensitive prefix match on the laptop name.
# Returns -1 if no match is found — the agent handles this gracefully.

from pathlib import Path
import pandas as pd

# Adjust this path if running outside the repo root
_repo_root = Path(".").resolve()
# Walk up until we find the data/ directory
for _parent in [_repo_root] + list(_repo_root.parents):
    if (_parent / "data" / "product").exists():
        _repo_root = _parent
        break

PRICING_CSV = _repo_root / "data" / "product" / "Laptop pricing.csv"
PDF_PATH    = _repo_root / "data" / "product" / "Laptop product descriptions.pdf"

if PRICING_CSV.exists():
    product_pricing_df = pd.read_csv(PRICING_CSV)
    print(f"Loaded {len(product_pricing_df)} laptops from CSV")
    print(product_pricing_df.head())
else:
    print(f"CSV not found at {PRICING_CSV}")
    print("Using a mock DataFrame for the notebook demo.")
    product_pricing_df = pd.DataFrame({
        "Name": ["SpectraBook Pro", "GammaAir X", "OmegaPro 15", "AcmeRight 12"],
        "Price": [1499, 999, 1299, 799],
    })
    print(product_pricing_df)


@tool
def get_laptop_price(laptop_name: str) -> int:
    """
    Return the price of a laptop given its name.
    Performs a prefix match (case-insensitive) against the product catalog.
    Returns the price as an integer if found, or -1 if no matching laptop exists.
    Use this tool when the user asks about the cost or price of a laptop.
    """
    matches = product_pricing_df[
        product_pricing_df["Name"].str.contains("^" + laptop_name, case=False)
    ]
    if matches.empty:
        return -1
    return int(matches["Price"].iloc[0])


# Smoke test
print("\nPrice of SpectraBook:", get_laptop_price.invoke({"laptop_name": "SpectraBook"}))
print("Price of NonExistent:", get_laptop_price.invoke({"laptop_name": "NonExistent"}))

In [ ]:
# ─── 7-B: PDF retrieval tool via ChromaDB ────────────────────────────────────
# Loads the product descriptions PDF, splits it into chunks,
# embeds into ChromaDB, and wraps it as a LangChain retriever tool.
#
# create_retriever_tool converts any LangChain retriever into a @tool-compatible
# tool the agent can call. The name and description become the tool schema.

from langchain.tools.retriever import create_retriever_tool
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

embedding = OpenAIEmbeddings(model="text-embedding-3-small")

if PDF_PATH.exists():
    print(f"Loading PDF: {PDF_PATH}")
    loader = PyPDFLoader(str(PDF_PATH))
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=256)
    splits = splitter.split_documents(docs)
    vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)
    print(f"Indexed {len(splits)} chunks from PDF")
else:
    print(f"PDF not found at {PDF_PATH}")
    print("Creating a mock vectorstore with sample product descriptions.")
    mock_docs = [
        Document(page_content="SpectraBook Pro: 12-core Intel CPU, 32GB RAM, 1TB SSD, 15-inch OLED display. Designed for creative professionals."),
        Document(page_content="GammaAir X: Ultra-thin 1.1kg design, 10-core ARM CPU, 16GB RAM, 14-inch IPS display. Best-in-class battery life."),
        Document(page_content="OmegaPro 15: 16-core AMD Ryzen CPU, 64GB RAM, 2TB NVMe. Workstation-grade performance for engineers."),
        Document(page_content="AcmeRight 12: Entry-level laptop, 8-core CPU, 8GB RAM, 256GB SSD. Perfect for students and everyday tasks."),
    ]
    vectorstore = Chroma.from_documents(documents=mock_docs, embedding=embedding)
    print(f"Mock vectorstore ready with {len(mock_docs)} documents")

get_product_features = create_retriever_tool(
    vectorstore.as_retriever(search_kwargs={"k": 1}),
    name="Get_Product_Features",
    description=(
        "Search for laptop product features, specifications, and descriptions. "
        "Use this tool when the user asks about a laptop's features, CPU, RAM, "
        "storage, display, or design. Input: the laptop name to search for."
    ),
)

print("\nRetriever tool ready:", get_product_features.name)

In [ ]:
# ─── 7-C: Build and run the Product Q&A agent ─────────────────────────────────

qna_system_prompt = SystemMessage("""
    You are a professional chatbot that answers questions about laptops sold by our company.
    To answer questions about laptops, you will ONLY use the available tools — not your own memory.
    If a tool returns -1 for a price, tell the user the laptop was not found in the catalog.
    Handle greetings and small talk with brief, professional responses.
""")

qna_agent = create_react_agent(
    model=llm,
    tools=[get_laptop_price, get_product_features],
    prompt=qna_system_prompt,
    checkpointer=MemorySaver(),
)

# Run a multi-turn product Q&A session
session_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

questions = [
    "Hello!",
    "I'm looking to buy a laptop — what do you have?",
    "Tell me about the features of SpectraBook.",
    "How much does it cost?",
    "What about GammaAir?",
]

for q in questions:
    print(f"\nUSER: {q}")
    response = qna_agent.invoke(
        {"messages": [HumanMessage(q)]},
        session_config,
    )
    print(f"AGENT: {response['messages'][-1].content}")
    print("─" * 60)

In [ ]:
# ─── 7-D: Stream the Q&A agent to watch tool calls live ───────────────────────

stream_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("Q: What are the features and pricing for OmegaPro?\n")
print("--- Streaming trace ---")
for state in qna_agent.stream(
    {"messages": [HumanMessage("What are the features and pricing for OmegaPro?")]},
    stream_config,
    stream_mode="values",
):
    msg = state["messages"][-1]
    role = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  [{role}] → {tc['name']}({tc['args']})")
    elif isinstance(msg, ToolMessage):
        print(f"  [ToolMessage] → {str(msg.content)[:120]}")
    elif isinstance(msg, AIMessage) and msg.content:
        print(f"  [AIMessage (final)] → {msg.content[:200]}")

---

## Part 8 — Debugging: Common Failures and How to Diagnose Them

---

### Common Failure Modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|-----------|-----|
| **Wrong tool selected** | Agent calls `find_product` when asked to add | Vague docstrings | Rewrite docstring to specify the exact use case |
| **No tool called** | Agent answers from its own training | Tool schema doesn't match the query intent | Improve docstring; add examples |
| **Infinite loop** | Agent keeps calling tools, never answers | Tool returns confusing results | Add guard: check `recursion_limit` |
| **Memory leak** | Old conversation context interferes | Wrong `thread_id` reuse | Use `uuid.uuid4()` for each new session |
| **Tool error not handled** | Agent crashes on `-1` or empty result | No fallback in tool | Return a human-readable string instead of raw error |

In [ ]:
# ─── 8-A: Inspect full agent state ───────────────────────────────────────────
# After .invoke(), you can inspect every message in the thread.
# This is the primary debugging tool.

def print_agent_trace(messages: list) -> None:
    """Pretty-print a full agent message trace with message types and roles."""
    print(f"{'─'*70}")
    print(f"Total messages in trace: {len(messages)}")
    print(f"{'─'*70}")
    for i, msg in enumerate(messages):
        role = type(msg).__name__
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [{i:02d}] {role:<15}  TOOL_CALL  {tc['name']}({tc['args']})")
        elif isinstance(msg, ToolMessage):
            content_preview = str(msg.content)[:80].replace("\n", " ")
            print(f"  [{i:02d}] {role:<15}  RESULT     {content_preview}")
        else:
            content_preview = str(msg.content)[:80].replace("\n", " ") if msg.content else ""
            print(f"  [{i:02d}] {role:<15}  {content_preview}")
    print(f"{'─'*70}")


debug_config = {"configurable": {"thread_id": "debug-trace"}}
debug_result = math_agent.invoke(
    {"messages": [HumanMessage("What is 5 plus 3, then times 4?")]},
    debug_config,
)
print_agent_trace(debug_result["messages"])

In [ ]:
# ─── 8-B: Docstring quality — bad vs good ────────────────────────────────────
# Demonstrates how a vague docstring causes the agent to pick the wrong tool.

@tool
def bad_sum(x: int, y: int) -> int:
    """Do math."""  # vague — LLM may not know when to use this
    return x + y

@tool
def bad_product(x: int, y: int) -> int:
    """Math operation."""  # equally vague
    return x * y

bad_agent = create_react_agent(
    model=llm,
    tools=[bad_sum, bad_product],
    prompt=SystemMessage("You are a math assistant. Use tools for calculations."),
)

bad_result = bad_agent.invoke(
    {"messages": [HumanMessage("Add 10 and 20")]},
    {"configurable": {"thread_id": "docstring-test"}},
)
print("Agent with BAD docstrings:")
print_agent_trace(bad_result["messages"])
print()
print("Note: The LLM may compute the answer itself (ignoring tools) or call the wrong tool.")
print("Compare with math_agent (good docstrings) — it always uses the correct tool.")

---

## Exercises

---

### Exercise 1 — Add a Division Tool

Create `@tool divide(x: int, y: int) -> float` that divides `x` by `y`.
- Handle the case where `y == 0` (return an error string, not an exception)
- Add it to the math agent and ask: `"What is 100 divided by 4, then times 3?"`
- Verify the agent chains `divide` and `find_product` in the correct order

### Exercise 2 — Docstring Matters

Change `find_sum`'s docstring to `"Look up weather for a city"`. Ask `"What is 5 + 3?"`. What does the agent do? What does this reveal about how tool selection works?

### Exercise 3 — Without Memory

Remove `checkpointer=MemorySaver()` from `create_react_agent`. Repeat the multi-turn test from Part 6: ask `"What is 7 plus 8?"` then `"Multiply that result by 3."` Does the agent remember Turn 1?

### Exercise 4 — Add a Subtraction Tool and Test Error Handling

Create `@tool find_difference(x: int, y: int) -> int`. Ask `"What is 100 minus 37?"`. Then ask `"Is that result positive or negative?"` — this tests whether the agent correctly reasons about the tool output without calling another tool.

In [ ]:
# ── Exercise 1 Starter — Division tool ───────────────────────────────────────

@tool
def divide(x: int, y: int) -> float:
    """
    TODO: write a docstring that clearly explains when the LLM should call this tool.
    """
    # TODO: implement division with zero-division guard
    pass


# TODO: rebuild math_agent with divide added to the tools list
# extended_agent = create_react_agent(
#     model=llm,
#     tools=[find_sum, find_product, divide],
#     prompt=system_prompt,
#     checkpointer=MemorySaver(),
# )

# TODO: test with a two-step query
# result = extended_agent.invoke(
#     {"messages": [HumanMessage("What is 100 divided by 4, then times 3?")]},
#     {"configurable": {"thread_id": "ex1"}},
# )
# print(result["messages"][-1].content)

In [ ]:
# ── Exercise 3 Starter — No memory ───────────────────────────────────────────

# TODO: build an agent WITHOUT checkpointer=MemorySaver()
# stateless_agent = create_react_agent(
#     model=llm,
#     tools=[find_sum, find_product],
#     prompt=system_prompt,
# )

# TODO: run two turns with the same thread_id and observe
# r1 = stateless_agent.invoke(
#     {"messages": [HumanMessage("What is 7 plus 8?")]},
#     {"configurable": {"thread_id": "no-mem"}},
# )
# print("Turn 1:", r1["messages"][-1].content)
#
# r2 = stateless_agent.invoke(
#     {"messages": [HumanMessage("Multiply that result by 3.")]},
#     {"configurable": {"thread_id": "no-mem"}},
# )
# print("Turn 2:", r2["messages"][-1].content)  # should ask for clarification

---

## What's Next?

You now understand the full ReAct loop, how `create_react_agent` assembles it, and how to add tools, memory, and streaming to a real agent. Here's where to go deeper:

### Build the loop manually
- **Example 1 — basic-local-rag** ([`../1-basic-local-rag/`](../1-basic-local-rag/)): Build the same `agent → tools` cycle from scratch with `StateGraph`, `add_node`, and `add_conditional_edges`. `create_react_agent` is this graph pre-assembled — seeing the internals is essential before customizing it.

### Multi-agent patterns
- **Example 5 — react-agent-lg** ([`../5-react-agent-lg/`](../5-react-agent-lg/)): A two-agent critic loop — a summarizer agent and a reviewer agent check each other's work. Demonstrates how to wire multiple `create_react_agent` instances together.

### Production-grade retrieval agents
- **Example 30 — agentic-rag** ([`../30-agentic-rag/`](../30-agentic-rag/)): `create_react_agent` with retrieval tools (ChromaDB), web search, and a calculator. The natural next step after the product Q&A agent in Part 7.

### Evaluate what you built
- **Example 16 — RAG Evaluation** ([`../16-rag-eval-notebook/rag_eval_workbook.ipynb`](../16-rag-eval-notebook/rag_eval_workbook.ipynb)): Score retrieval quality with RAGAS (Faithfulness, Answer Relevance, Context Recall).

### Further reading
- Yao et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629
- Wei et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models.* NeurIPS 2022. https://arxiv.org/abs/2201.11903
- LangGraph conceptual guide — agent architectures: https://langchain-ai.github.io/langgraph/concepts/agentic_concepts/
- LangGraph `create_react_agent` reference: https://langchain-ai.github.io/langgraph/reference/prebuilt/

---

*Workshop complete. Open example 5 to see two ReAct agents working together.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself.

---

### Exercise 1 — Add a Division Tool

**Key insight:** The zero-division guard must return a *string* (not raise an exception) so the LLM can read the error message and respond to the user gracefully. If the tool raises, the agent crashes.

**Expected trace for `"What is 100 divided by 4, then times 3?"`:**
1. `AIMessage` → `tool_calls: divide(100, 4)`
2. `ToolMessage` → `25.0`
3. `AIMessage` → `tool_calls: find_product(25, 3)`
4. `ToolMessage` → `75`
5. `AIMessage` → `"The answer is 75."`

In [ ]:
# Answer — Exercise 1

@tool
def divide_ans(x: int, y: int) -> float | str:
    """
    Divide x by y and return the result as a float.
    Use this tool whenever the user needs to compute the division or quotient of two numbers.
    Returns an error string if y is zero.
    """
    if y == 0:
        return "Error: cannot divide by zero"
    return x / y


extended_agent = create_react_agent(
    model=llm,
    tools=[find_sum, find_product, divide_ans],
    prompt=SystemMessage(
        "You are a Math assistant. Solve problems using ONLY the available tools."
    ),
    checkpointer=MemorySaver(),
)

ext_result = extended_agent.invoke(
    {"messages": [HumanMessage("What is 100 divided by 4, then times 3?")]},
    {"configurable": {"thread_id": "ans1"}},
)
print_agent_trace(ext_result["messages"])
print("Answer:", ext_result["messages"][-1].content)

# Zero-division guard test
zero_result = extended_agent.invoke(
    {"messages": [HumanMessage("What is 10 divided by 0?")]},
    {"configurable": {"thread_id": "ans1-zero"}},
)
print("\nZero division:", zero_result["messages"][-1].content)

### Exercise 2 — Docstring Matters

**Expected behavior:** With the docstring changed to `"Look up weather for a city"`, the LLM may:
- Compute `5 + 3 = 8` itself (from training knowledge) without calling any tool
- Call `find_sum` anyway if the tool name is still suggestive, but be confused about arguments
- Ask the user for a city name (following the docstring literally)

**Key insight:** The LLM does not look at the function body or name — only the docstring and the argument schema. A tool called `find_sum` with the docstring `"Look up weather"` will be treated as a weather tool.

In [ ]:
# Answer — Exercise 2

@tool
def find_sum_misleading(x: int, y: int) -> int:
    """Look up weather for a city."""  # deliberately wrong docstring
    return x + y

misleading_agent = create_react_agent(
    model=llm,
    tools=[find_sum_misleading],
    prompt=SystemMessage("You are a helpful assistant."),
)

m_result = misleading_agent.invoke(
    {"messages": [HumanMessage("What is 5 + 3?")]},
    {"configurable": {"thread_id": "misleading-docstring"}},
)
print("Agent with misleading docstring:")
print_agent_trace(m_result["messages"])
print()
print("Observation: The LLM likely answers without calling the tool,")
print("or calls it incorrectly because the docstring says 'city', not numbers.")

### Exercise 3 — Without Memory

**Expected behavior:** Without `MemorySaver`, each `.invoke()` call is stateless. On Turn 2 (`"Multiply that result by 3"`), the agent has no knowledge of Turn 1. It will likely:
- Ask "What result are you referring to?"
- Or attempt to call a tool with an undefined argument

**Key insight:** `thread_id` alone does not create memory — it is just a key. Memory requires a `checkpointer` that actually persists state. Without `checkpointer=MemorySaver()`, `thread_id` is ignored.

In [ ]:
# Answer — Exercise 3

stateless_agent = create_react_agent(
    model=llm,
    tools=[find_sum, find_product],
    prompt=SystemMessage(
        "You are a Math assistant. Solve problems using ONLY the available tools."
    ),
    # No checkpointer — stateless
)

r1 = stateless_agent.invoke(
    {"messages": [HumanMessage("What is 7 plus 8?")]},
    {"configurable": {"thread_id": "no-mem"}},
)
print("Turn 1:", r1["messages"][-1].content)

r2 = stateless_agent.invoke(
    {"messages": [HumanMessage("Multiply that result by 3.")]},
    {"configurable": {"thread_id": "no-mem"}},
)
print("Turn 2 (no memory):", r2["messages"][-1].content)
print()
print("Without MemorySaver, Turn 2 has no context from Turn 1.")
print("The agent cannot know 'that result' refers to 15.")

### Exercise 4 — Subtraction Tool and Reasoning

**Expected behavior:** After `find_difference(100, 37) → 63`, asking `"Is that result positive or negative?"` should be answered **without a tool call** — the agent should reason from the previous ToolMessage in the thread history.

**Key insight:** The LLM can reason about tool results already in the conversation without calling another tool. Agents don't need a "check sign" tool — they can read and reason about previous outputs. This is the "Observe" phase of ReAct.

In [ ]:
# Answer — Exercise 4

@tool
def find_difference(x: int, y: int) -> int:
    """
    Subtract y from x and return the difference.
    Use this tool whenever the user needs to compute the difference,
    subtraction, or 'minus' operation of two integers.
    """
    return x - y


sub_agent = create_react_agent(
    model=llm,
    tools=[find_sum, find_product, divide_ans, find_difference],
    prompt=SystemMessage(
        "You are a Math assistant. Solve problems using ONLY the available tools. "
        "When asked a yes/no question about a prior result, answer from the conversation history."
    ),
    checkpointer=MemorySaver(),
)

sub_config = {"configurable": {"thread_id": "subtraction-test"}}

r1 = sub_agent.invoke(
    {"messages": [HumanMessage("What is 100 minus 37?")]},
    sub_config,
)
print("Turn 1:", r1["messages"][-1].content)

r2 = sub_agent.invoke(
    {"messages": [HumanMessage("Is that result positive or negative?")]},
    sub_config,
)
print("Turn 2:", r2["messages"][-1].content)
print()

# Check: Turn 2 should have NO tool calls — it reasons from the prior result
last_ai = [m for m in r2["messages"] if isinstance(m, AIMessage)][-1]
made_tool_call = bool(getattr(last_ai, "tool_calls", []))
print(f"Turn 2 made a tool call: {made_tool_call}")
print("Expected: False — the agent should answer from the existing conversation context.")